# Baseline-прогноз спроса по месяцам
## Контекст:

В рамках проекта автоматизации заказов для магазина конфет были:

- собраны помесячные данные продаж
- проведён XYZ-анализ ассортимента
- определены товары, допустимые к прогнозированию (X и Y)



Цель данного ноутбука — реализовать baseline-прогноз спроса, оценить его качество и определить границы применимости автоматизации.


## Цель и задачи этапа
- Реализовать простые и интерпретируемые baseline-прогнозы
- Проверить прогнозируемость данных
- Получить эталон для сравнения с ML-моделями
- Подготовить основу для расчёта заказов

## Описание входных данных
Используемые данные:
- помесячные продажи товаров
- XYZ-классификация ассортимента (операционная версия)

Прогнозирование выполняется только для товаров классов X и Y.

## Загрузка и первичная проверка данных
- Загрузка данных продаж
- Загрузка XYZ-классификации
- Проверка структуры и типов данных

In [1]:
import pandas as pd
from pathlib import Path
from IPython.display import display

data_path = Path.cwd().parent.parent / "data" / "clean"
sales_data_path = data_path / "sales_data.csv"
xyz_operational_path = data_path / "xyz_operational_result.csv"

sales_data = pd.read_csv(sales_data_path)
xyz_data = pd.read_csv(xyz_operational_path)

display(sales_data.head(), xyz_data.head())

,product,sku,qty,unit,month
0,КОНФ ВЕС Столичные,КО01828,96.613,кг,2025-08
1,КОНФ ВЕС Сибирский Сувенир,НС07823,31.901,кг,2025-08
2,КОНФ ВЕС Тамбовский волк Люкс,ТК07914,26.931,кг,2025-08
3,КОНФ ВЕС БУТЫЛОЧКИ С КОНЬЯКОМ,ЯП24671,22.990,кг,2025-08
4,КОНФ ВЕС Столичные любимые,ВО13951,39.748,кг,2025-08


,Артикул,Количество месяцев,CV коэффициент,Класс товара
0,0104,1,NaN,Unknown
1,15033,10,0.372647,Y - Средний
2,38010,11,0.635256,Z - Нестабильный
3,38015,11,0.367171,Y - Средний
4,38030,10,0.293972,Y - Средний


## Отбор товаров для прогнозирования (XYZ-фильтр)
- Объединение продаж с XYZ-классами
- Исключение товаров классов Z и Unknown
- Анализ количества SKU до и после фильтрации

In [2]:
xyz_data = xyz_data[
    (xyz_data["Количество месяцев"] >= 3) & 
    (xyz_data["Класс товара"].isin(["X - Стабильный", "Y - Средний"]))]


display(xyz_data.value_counts("Класс товара"))
print(f"Всего товаров в выборке: {len(xyz_data)}")

Класс товара
Y - Средний       281
X - Стабильный     49
Name: count, dtype: int64

Всего товаров в выборке: 330


## Подготовка временных рядов продаж
- Агрегация продаж до формата:
- SKU
- месяц
- количество продаж
- Сортировка данных по времени
- Проверка полноты временной истории
- Исключение товаров с недостаточной историей

In [3]:
xyz_data.sort_values("Количество месяцев", ascending=False)

,Артикул,Количество месяцев,CV коэффициент,Класс товара
454,РФ18209,11,0.280380,Y - Средний
199,КО17619,11,0.309199,Y - Средний
178,КО14968,11,0.445065,Y - Средний
181,КО16060,11,0.325219,Y - Средний
446,РФ17657,11,0.260968,Y - Средний
...,...,...,...,...
669,ЯП06155,3,0.428741,Y - Средний
412,РФ13537,3,0.045580,X - Стабильный
638,ТК24044,3,0.482183,Y - Средний
286,КО25454,3,0.450694,Y - Средний


## Выбор схемы валидации (backtesting)
- Определение обучающего периода
- Определение тестового периода
- Обоснование выбранной схемы валидации

In [4]:
train_end_date = "2025-10"
test_month = "2025-11"

test_df = sales_data[sales_data["month"] == test_month]
train_df = sales_data[sales_data["month"] <= train_end_date].sort_values("month", ascending=True)
display(test_df.head(), train_df.head())

,product,sku,qty,unit,month
4452,КОНФ ВЕС Столичные,КО01828,117.507,кг,2025-11
4453,КОНФ ВЕС БУТЫЛОЧКИ С КОНЬЯКОМ,ЯП24671,58.066,кг,2025-11
4454,КОНФ ВЕС Тамбовский волк Люкс,ТК07914,39.632,кг,2025-11
4455,КОНФ ВЕС Столичные любимые,ВО13951,58.934,кг,2025-11
4456,КОНФ ВЕС Сибирский Сувенир,НС07823,15.910,кг,2025-11


,product,sku,qty,unit,month
5124,КОНФ ВЕС ЦЕНТР ДЕРЖАВЫ,НС02130,2.861,кг,2025-01
5171,Карамель Клюквенная РОТ ФРОНТ 1 кг,ББ08671,4.196,кг,2025-01
5172,Карамель Миньон РОТ ФРОНТ,РФ18742,4.036,кг,2025-01
5173,КОНФ ВЕС Сибирский сувенир Кедровый грильяж,НС20906,0.453,кг,2025-01
5174,КОНФ ВЕС Батончики Ореховая роща КРАСНЫЙ ОКТЯБРЬ,КО17598,3.316,кг,2025-01


## Baseline-прогноз №1: Last Month
- Описание метода
- Логика расчёта прогноза
- Ограничения и ожидаемое поведение

In [5]:
last_month_forecast = train_df[
    (train_df["sku"].isin(xyz_data["Артикул"])) &
    (train_df["month"] == train_end_date)
].reset_index(drop=True)

display(last_month_forecast.head())

,product,sku,qty,unit,month
0,"Вафли NEO-BOTANICA No sugar added ""Вафельные х...",РФ25177,28.0,шт,2025-10
1,Вафли Артек 200 гр РОТ ФРОНТ,РФ11926,17.0,шт,2025-10
2,Печенье Аленка сдобное хрустящее с клюквой и ш...,ЯП24446,15.0,шт,2025-10
3,Печенье АЛЕНКА с начинкой со вкусом экзотика 1...,ЯП25898,35.0,шт,2025-10
4,Зефир NEO-BOTANICA смузи абрикос-манго 280 гр,РФ25111,10.0,шт,2025-10


## Baseline-прогноз №2: Moving Average
- Описание метода
- Выбор размера окна
- Сравнение разных вариантов скользящего среднего

ИСПРАВИТЬ ДЛЯ ТОВАРОВ С НЕДОСТАТОЧНЫМ КОЛИЧЕСТВОМ МЕСЯЦЕВ!!!!

In [6]:
ma_window = 6

sales_xy = train_df[train_df["sku"].isin(xyz_data["Артикул"])].sort_values(["sku", "month"])

moving_avg_forecast = (
    sales_xy
    .groupby("sku", as_index=False)
    .tail(ma_window)
    .groupby("sku", as_index=False)
    .agg(forecast_qty=("qty", "mean"))
)

moving_avg_forecast["month"] = test_month

# Переставим колонки для удобства
moving_avg_forecast = moving_avg_forecast[["sku", "month", "forecast_qty"]]

display(moving_avg_forecast.head())

,sku,month,forecast_qty
0,15033,2025-11,7.000000
1,38015,2025-11,12.833333
2,38030,2025-11,5.500000
3,9127275,2025-11,34.166667
4,ББ00176,2025-11,94.666667


## Формирование таблицы прогнозов
- Объединение прогнозов разных baseline-методов
- Подготовка единого датафрейма для оценки качества

In [7]:
lm_forecast_nov = last_month_forecast.copy()
lm_forecast_nov["forecast_qty"] = lm_forecast_nov["qty"]
lm_forecast_nov["month"] = test_month
lm_forecast_nov = lm_forecast_nov[["sku", "month", "forecast_qty"]]
avg_foregast_nov = moving_avg_forecast.copy()

display(lm_forecast_nov.head(), avg_foregast_nov.head())

,sku,month,forecast_qty
0,РФ25177,2025-11,28.0
1,РФ11926,2025-11,17.0
2,ЯП24446,2025-11,15.0
3,ЯП25898,2025-11,35.0
4,РФ25111,2025-11,10.0


,sku,month,forecast_qty
0,15033,2025-11,7.000000
1,38015,2025-11,12.833333
2,38030,2025-11,5.500000
3,9127275,2025-11,34.166667
4,ББ00176,2025-11,94.666667


## Оценка качества baseline-прогнозов
- Сравнение прогнозов с фактическими продажами
- Расчёт метрик качества (MAE, MAPE)
- Анализ ошибок по товарам

In [11]:
actual_nov = test_df[["sku", "month", "qty"]].rename(columns={"qty": "actual_qty"})

lm_metrics_df = (
    lm_forecast_nov.rename(columns={"forecast_qty": "lm_forecast_qty"})
    .merge(actual_nov, on=["sku", "month"], how="left")
)
lm_metrics_df["MAE"] = (lm_metrics_df["actual_qty"] - lm_metrics_df["lm_forecast_qty"]).abs()
lm_metrics_df["MAPE"] = (
    lm_metrics_df["MAE"]
    .div(lm_metrics_df["actual_qty"].abs().where(lm_metrics_df["actual_qty"] != 0))
    .mul(100)
)
lm_metrics_df = lm_metrics_df[["sku", "month", "lm_forecast_qty", "MAE", "MAPE"]].sort_values("sku")

avg_metrics_df = (
    avg_foregast_nov.rename(columns={"forecast_qty": "avg_foregast_qty"})
    .merge(actual_nov, on=["sku", "month"], how="left")
)
avg_metrics_df["MAE"] = (avg_metrics_df["actual_qty"] - avg_metrics_df["avg_foregast_qty"]).abs()
avg_metrics_df["MAPE"] = (
    avg_metrics_df["MAE"]
    .div(avg_metrics_df["actual_qty"].abs().where(avg_metrics_df["actual_qty"] != 0))
    .mul(100)
)
avg_metrics_df = avg_metrics_df[["sku", "month", "avg_foregast_qty", "MAE", "MAPE"]].sort_values("sku")

output_path = Path.cwd().parent.parent / "data" / "clean"
lm_metrics_df.to_csv(output_path / "lm_nov_metrics.csv", index=False)
avg_metrics_df.to_csv(output_path / "avg_nov_metrics.csv", index=False)

display(lm_metrics_df.head(), avg_metrics_df.head(), test_df.head())

,sku,month,lm_forecast_qty,MAE,MAPE
67,15033,2025-11,3.0,NaN,NaN
69,38015,2025-11,11.0,6.0,35.294118
68,38030,2025-11,8.0,1.0,14.285714
87,9127275,2025-11,26.0,3.0,13.043478
217,ББ00176,2025-11,143.0,59.0,29.207921


,sku,month,avg_foregast_qty,MAE,MAPE
0,15033,2025-11,7.000000,NaN,NaN
1,38015,2025-11,12.833333,4.166667,24.509804
2,38030,2025-11,5.500000,1.500000,21.428571
3,9127275,2025-11,34.166667,11.166667,48.550725
4,ББ00176,2025-11,94.666667,107.333333,53.135314


,product,sku,qty,unit,month
4452,КОНФ ВЕС Столичные,КО01828,117.507,кг,2025-11
4453,КОНФ ВЕС БУТЫЛОЧКИ С КОНЬЯКОМ,ЯП24671,58.066,кг,2025-11
4454,КОНФ ВЕС Тамбовский волк Люкс,ТК07914,39.632,кг,2025-11
4455,КОНФ ВЕС Столичные любимые,ВО13951,58.934,кг,2025-11
4456,КОНФ ВЕС Сибирский Сувенир,НС07823,15.910,кг,2025-11


## Сравнение baseline-методов
- Анализ различий между Last Month и Moving Average
- Поведение прогнозов для товаров X и Y
- Выявление систематических ошибок

In [9]:
class_map = (
    xyz_data[["Артикул", "Класс товара"]]
    .drop_duplicates()
    .rename(columns={"Артикул": "sku", "Класс товара": "xy_class"})
)

compare_df = (
    lm_metrics_df
    .merge(avg_metrics_df, on=["sku", "month"], how="inner", suffixes=("_lm", "_ma"))
    .merge(class_map, on="sku", how="left")
)

# В анализ сравнения берём только SKU, где есть факт и для LM, и для MA
valid_df = compare_df[compare_df["MAE_lm"].notna() & compare_df["MAE_ma"].notna()].copy()

overall_summary = pd.DataFrame({
    "method": ["Last Month", "Moving Average"],
    "mean_MAE": [valid_df["MAE_lm"].mean(), valid_df["MAE_ma"].mean()],
    "median_MAE": [valid_df["MAE_lm"].median(), valid_df["MAE_ma"].median()],
    "mean_MAPE": [valid_df["MAPE_lm"].mean(), valid_df["MAPE_ma"].mean()],
    "median_MAPE": [valid_df["MAPE_lm"].median(), valid_df["MAPE_ma"].median()],
})

by_class_summary = (
    valid_df.groupby("xy_class", dropna=False)
    .agg(
        sku_cnt=("sku", "nunique"),
        lm_mean_MAE=("MAE_lm", "mean"),
        ma_mean_MAE=("MAE_ma", "mean"),
        lm_mean_MAPE=("MAPE_lm", "mean"),
        ma_mean_MAPE=("MAPE_ma", "mean"),
    )
    .reset_index()
)

valid_df["best_method_by_sku"] = valid_df.apply(
    lambda x: "Last Month" if x["MAPE_lm"] < x["MAPE_ma"] else ("Moving Average" if x["MAPE_ma"] < x["MAPE_lm"] else "Tie"),
    axis=1,
)

wins = valid_df["best_method_by_sku"].value_counts()
lm_wins = int(wins.get("Last Month", 0))
ma_wins = int(wins.get("Moving Average", 0))
tie_wins = int(wins.get("Tie", 0))

no_fact_cnt = compare_df[compare_df["MAE_lm"].isna() | compare_df["MAE_ma"].isna()]["sku"].nunique()
hard_cases = valid_df[(valid_df["MAPE_lm"] > 100) & (valid_df["MAPE_ma"] > 100)].copy()
hard_share = (len(hard_cases) / len(valid_df) * 100) if len(valid_df) else 0

hard_cases_top = (
    hard_cases.assign(best_mape=hard_cases[["MAPE_lm", "MAPE_ma"]].min(axis=1))
    .sort_values("best_mape", ascending=False)
    [["sku", "xy_class", "MAE_lm", "MAPE_lm", "MAE_ma", "MAPE_ma"]]
    .head(10)
)

display(overall_summary, by_class_summary, hard_cases_top)

main_method = "Last Month" if overall_summary.loc[overall_summary["method"] == "Last Month", "mean_MAPE"].iloc[0] < overall_summary.loc[overall_summary["method"] == "Moving Average", "mean_MAPE"].iloc[0] else "Moving Average"

x_row = by_class_summary[by_class_summary["xy_class"] == "X - Стабильный"]
y_row = by_class_summary[by_class_summary["xy_class"] == "Y - Средний"]

x_comment = "нет данных по классу X" if x_row.empty else (
    f"в классе X лучше {'Last Month' if x_row['lm_mean_MAPE'].iloc[0] < x_row['ma_mean_MAPE'].iloc[0] else 'Moving Average'}"
)
y_comment = "нет данных по классу Y" if y_row.empty else (
    f"в классе Y лучше {'Last Month' if y_row['lm_mean_MAPE'].iloc[0] < y_row['ma_mean_MAPE'].iloc[0] else 'Moving Average'}"
)

print("1. Какой baseline в среднем лучше?")
print(
    f"   В среднем лучше {main_method}: mean MAPE = {overall_summary.loc[overall_summary['method'] == main_method, 'mean_MAPE'].iloc[0]:.2f}%. "
    f"По победам на SKU: Last Month={lm_wins}, Moving Average={ma_wins}, Tie={tie_wins}."
)
print("\n2. Отличается ли поведение для X и Y?")
print(f"   Да, поведение различается: {x_comment}; {y_comment}.")
print("\n3. Где baseline принципиально не работает?")
print(
    f"   Проблемные зоны: SKU без факта в ноябре ({no_fact_cnt} SKU, метрики NaN) и кейсы, где оба метода дают MAPE > 100% "
    f"({len(hard_cases)} SKU, {hard_share:.1f}% от валидной выборки)."
)
print("\n4. Какой метод логично взять как основной?")
print(
    f"   Логично взять {main_method} как основной baseline, а второй метод использовать как sanity-check/fallback для SKU, "
    "где основной метод систематически хуже."
)


,method,mean_MAE,median_MAE,mean_MAPE,median_MAPE
0,Last Month,9.571704,3.139,109.607081,27.272727
1,Moving Average,9.559882,3.597,112.739310,28.807217


,xy_class,sku_cnt,lm_mean_MAE,ma_mean_MAE,lm_mean_MAPE,ma_mean_MAPE
0,X - Стабильный,38,3.153789,3.127682,22.547793,18.739569
1,Y - Средний,205,10.761366,10.752192,125.744900,130.163652


,sku,xy_class,MAE_lm,MAPE_lm,MAE_ma,MAPE_ma
241,КО10441,Y - Средний,2.321,8926.923077,3.126833,12026.282051
267,КО01914,Y - Средний,2.569,2594.949495,3.313333,3346.801347
54,ВО24704,Y - Средний,30.000,1000.000000,24.000000,800.000000
257,КО18673,Y - Средний,133.000,950.000000,96.166667,686.904762
112,ВО00365,Y - Средний,9.160,771.693345,4.560500,384.203875
1,РФ11926,Y - Средний,14.000,466.666667,10.833333,361.111111
198,КО21237,Y - Средний,25.000,357.142857,25.500000,364.285714
40,КО05702,Y - Средний,10.000,500.000000,6.666667,333.333333
59,ПЗ21362,Y - Средний,16.000,320.000000,21.500000,430.000000
10,РФ25110,Y - Средний,11.000,366.666667,9.000000,300.000000


1. Какой baseline в среднем лучше?
   В среднем лучше Last Month: mean MAPE = 109.61%. По победам на SKU: Last Month=114, Moving Average=126, Tie=3.

2. Отличается ли поведение для X и Y?
   Да, поведение различается: в классе X лучше Moving Average; в классе Y лучше Last Month.

3. Где baseline принципиально не работает?
   Проблемные зоны: SKU без факта в ноябре (27 SKU, метрики NaN) и кейсы, где оба метода дают MAPE > 100% (19 SKU, 7.8% от валидной выборки).

4. Какой метод логично взять как основной?
   Логично взять Last Month как основной baseline, а второй метод использовать как sanity-check/fallback для SKU, где основной метод систематически хуже.


## Выводы по baseline-прогнозу
1. **Какой baseline в среднем лучше**
- Основным считаем метод, у которого ниже средний `MAPE` на общей выборке (`overall_summary`).
- Итоговый выбор подтверждаем не только средним, но и числом SKU-побед (`best_method_by_sku`).

2. **Отличается ли поведение для X и Y**
- По `by_class_summary` оцениваем метрики отдельно для `X - Стабильный` и `Y - Средний`.
- Для класса `X` обычно ожидается более предсказуемая динамика и ниже ошибка, для `Y` — выше чувствительность к колебаниям.

3. **Где baseline принципиально не работает**
- SKU без фактических продаж в тестовом месяце дают `NaN` в метриках при текущем определении `MAPE`.
- Критичные кейсы: когда **оба** baseline дают высокий `MAPE` (например, >100%), то есть не ловят динамику спроса.
- Также baseline слаб в товарах с резкими всплесками/провалами и структурными сдвигами между месяцами.

4. **Какой метод брать как основной**
- В проде логично брать метод-победитель по `mean_MAPE` как основной baseline.
- Второй baseline использовать как fallback/контрольный ориентир на SKU, где основной стабильно хуже.
- Для товаров из «проблемной зоны» (высокий `MAPE` у обоих методов) требуется переход к более сильной модели (ML/доп. признаки).